In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 6
INTERVAL = "5m"
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "MATICUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from optuna.pruners import MedianPruner
from functools import partial
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split
from models import tune_selected_features_only , make_bucket_table, fit_final_model

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")

features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 1,000


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2024-09-06 15:40:00+00:00,0.3662,0.3679,0.3660,0.3677,172085.1,2024-09-06 15:44:59.999000+00:00,63124.74949,184,109146.8,...,-0.900969,0.937752,0.347305,-1.0,-1.836970e-16,0.000000,0.000000,0.000000,NaN,NaN
1,2024-09-06 15:45:00+00:00,0.3679,0.3683,0.3666,0.3667,236756.7,2024-09-06 15:49:59.999000+00:00,87019.72975,342,81575.3,...,-0.900969,0.937752,0.347305,-1.0,-1.836970e-16,-0.000022,-0.000012,-0.000010,NaN,NaN
2,2024-09-06 15:50:00+00:00,0.3669,0.3670,0.3655,0.3668,141178.7,2024-09-06 15:54:59.999000+00:00,51695.07204,216,90499.5,...,-0.900969,0.937752,0.347305,-1.0,-1.836970e-16,-0.000025,-0.000018,-0.000008,NaN,NaN
3,2024-09-06 15:55:00+00:00,0.3671,0.3671,0.3655,0.3663,93002.4,2024-09-06 15:59:59.999000+00:00,34067.68869,162,48157.3,...,-0.900969,0.937752,0.347305,-1.0,-1.836970e-16,-0.000044,-0.000027,-0.000018,NaN,NaN
4,2024-09-06 16:00:00+00:00,0.3661,0.3665,0.3651,0.3654,99800.5,2024-09-06 16:04:59.999000+00:00,36492.59671,207,55007.2,...,-0.900969,0.937752,0.347305,-1.0,-1.836970e-16,-0.000089,-0.000045,-0.000044,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 921
[info] optuna train rows: 588
[info] valid rows:        148
[info] test rows:         185


In [9]:
results = tune_selected_features_only(
    model_type=MODEL_TYPE,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    n_trials=100,
    objective_metric="roc_auc",
    top_k=25,
)

print(results["selected_features"])
print(results["feature_importance"].head(30))

[I 2026-03-22 18:29:53,324] A new study created in memory with name: no-name-a91cf01c-b9e2-4457-9614-208c3b3fa705


[I 2026-03-22 18:29:53,399] Trial 0 finished with value: 0.4952511415525115 and parameters: {'n_estimators': 400, 'learning_rate': 0.09423875899553878, 'max_depth': 5, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 3, 'reg_lambda': 0.13066739238053282, 'scale_pos_weight': 1.3552977514729934}. Best is trial 0 with value: 0.4952511415525115.


[I 2026-03-22 18:29:53,441] Trial 1 finished with value: 0.5511415525114155 and parameters: {'n_estimators': 600, 'learning_rate': 0.07036510754376854, 'max_depth': 3, 'subsample': 0.9909729556485982, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_lambda': 0.23102018878452935, 'scale_pos_weight': 0.8077137511852146}. Best is trial 1 with value: 0.5511415525114155.


[I 2026-03-22 18:29:53,490] Trial 2 finished with value: 0.6002739726027397 and parameters: {'n_estimators': 400, 'learning_rate': 0.05642937488667569, 'max_depth': 4, 'subsample': 0.7873687420594125, 'colsample_bytree': 0.8447411578889518, 'min_child_weight': 3, 'reg_lambda': 0.3839629299804172, 'scale_pos_weight': 0.9278748298829654}. Best is trial 2 with value: 0.6002739726027397.


[I 2026-03-22 18:29:53,539] Trial 3 finished with value: 0.48968036529680364 and parameters: {'n_estimators': 500, 'learning_rate': 0.0772099153368949, 'max_depth': 3, 'subsample': 0.8542703315240835, 'colsample_bytree': 0.836965827544817, 'min_child_weight': 2, 'reg_lambda': 1.6409286730647923, 'scale_pos_weight': 0.799865724974628}. Best is trial 2 with value: 0.6002739726027397.


...

[I 2026-03-22 18:29:59,658] Trial 96 finished with value: 0.6789041095890411 and parameters: {'n_estimators': 600, 'learning_rate': 0.07473961221539645, 'max_depth': 6, 'subsample': 0.8032477570825004, 'colsample_bytree': 0.8024642520877469, 'min_child_weight': 3, 'reg_lambda': 0.11297742244753478, 'scale_pos_weight': 0.8734251725279815}. Best is trial 67 with value: 0.6789041095890411.


[I 2026-03-22 18:29:59,730] Trial 97 finished with value: 0.6147945205479451 and parameters: {'n_estimators': 700, 'learning_rate': 0.07454499419111245, 'max_depth': 6, 'subsample': 0.8011014951526868, 'colsample_bytree': 0.8010423820925503, 'min_child_weight': 3, 'reg_lambda': 0.10000041904173389, 'scale_pos_weight': 0.8790515895124209}. Best is trial 67 with value: 0.6789041095890411.


[I 2026-03-22 18:29:59,797] Trial 98 finished with value: 0.6383561643835617 and parameters: {'n_estimators': 600, 'learning_rate': 0.08074542918913627, 'max_depth': 6, 'subsample': 0.8284000076374172, 'colsample_bytree': 0.7212541071183913, 'min_child_weight': 4, 'reg_lambda': 0.12890941946259185, 'scale_pos_weight': 0.8909695197934432}. Best is trial 67 with value: 0.6789041095890411.


[I 2026-03-22 18:29:59,862] Trial 99 pruned. 


['is_trending', 'atr_norm', 'dow_sin', 'trend_strength', 'range_15', 'vol_15', 'is_high_vol', 'hour_cos', 'trend_x_imb', 'dist_ma_30', 'vol_5', 'hour_sin', 'mom_60', 'dist_ma_15', 'imbalance_15', 'dist_ma_5', 'mom_x_imb', 'imbalance_5', 'range_5', 'vol_30', 'mom_5', 'vol_ratio_5_30', 'mr_x_vol', 'num_trades_mom_5', 'volume_mom_5']
feature
is_trending         5.724646
atr_norm            4.233803
dow_sin             3.531731
trend_strength      3.303753
range_15            3.181080
vol_15              3.154899
is_high_vol         3.113093
hour_cos            3.060222
trend_x_imb         2.994357
dist_ma_30          2.983027
vol_5               2.900199
hour_sin            2.825455
mom_60              2.684694
dist_ma_15          2.528436
imbalance_15        2.494826
dist_ma_5           2.460884
mom_x_imb           2.436568
imbalance_5         2.372575
range_5             2.371033
vol_30              2.367197
mom_5               2.360944
vol_ratio_5_30      2.220016
mr_x_vol            2

In [10]:
artifacts = fit_final_model(
    model_type=MODEL_TYPE,
    best_params=results["best_params"],
    selected_features=results["selected_features"],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

calibrator = artifacts["calibrator"]
base_model = artifacts["base_model"]
selected_features = artifacts["selected_features"]

In [11]:
X_train_sel = X_train[selected_features].copy()
X_valid_sel = X_valid[selected_features].copy()
X_train_full_sel = pd.concat([X_train_sel, X_valid_sel], axis=0)

X_test_sel = X_test[selected_features].copy()
y_train_full = pd.concat([y_train, y_valid], axis=0)

calibrator = artifacts["calibrator"]

train_pred = calibrator.predict_proba(X_train_full_sel)[:, 1]
test_pred = calibrator.predict_proba(X_test_sel)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train ROC AUC:   0.933841
Test ROC AUC:    0.581905
Train PR AUC:    0.919972
Test PR AUC:     0.522724
Train Log Loss:  0.685568
Test Log Loss:   0.694199
Train Brier:     0.246211
Test Brier:      0.250526
Train Accuracy:  0.527174
Test Accuracy:   0.432432
Train Precision: 0.527174
Test Precision:  0.432432
Train Recall:    1.000000
Test Recall:     1.000000
Train F1:        0.690391
Test F1:         0.603774


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret": fwd_ret.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret"].agg(["mean", "count", "std"])
print(bucket_stats)

                      mean  count       std
pred_bin                                   
(0.501, 0.5017]  -0.000736     19  0.004133
(0.5017, 0.5032] -0.000089     18  0.004837
(0.5032, 0.5042]  0.001739     19  0.005073
(0.5042, 0.5058]  0.001233     18  0.003969
(0.5058, 0.5072]  0.001441     19  0.004875
(0.5072, 0.5082] -0.000192     18  0.003937
(0.5082, 0.5092] -0.000964     18  0.003090
(0.5092, 0.5099]  0.000672     19  0.005344
(0.5099, 0.5103] -0.000470     18  0.002330
(0.5103, 0.5106] -0.000206     19  0.003697


/tmp/ipykernel_1029061/1883822384.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret"].mean())
overall_mean_ret = float(eval_df["fwd_ret"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret"].mean())

In [15]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/MATICUSDT__6_predictions.csv


In [16]:
# save model
joblib.dump(artifacts, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(selected_features, f, indent=2)

# save feature importance
results["feature_importance"].to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(results["study"].best_value),
    "model_params": results["best_params"],
    "n_features": int(len(selected_features)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/MATICUSDT__h6_model.joblib
[saved] features -> models/xgb/MATICUSDT__h6_feature_cols.json
[saved] feature importance -> models/xgb/MATICUSDT__h6_feature_importance.csv
[saved] metadata -> models/xgb/MATICUSDT__h6_meta.json
